# Gemini API 기반 이미지 생성 프롬프트 엔지니어링 실습

**💡 사전 필수 준비 (API Key 설정)**
1. Google Colab 왼쪽 사이드바에서 **'열쇠 모양 아이콘'(보안 비밀)**을 클릭합니다.
2. 새 비밀 추가를 눌러 이름에 `GEMINI_API_KEY`를 넣고, 값에 발급받은 API 키를 붙여넣습니다.
3. 등록한 키 옆의 **'노트북 액세스' 토글 버튼을 켜서 활성화**해야 코드가 정상 작동합니다.

## 제1장. 강의 개요 및 개발 환경 구축

In [ ]:
# 라이브러리 설치 (최신버전 유지)
!pip install -q -U google-genai pillow

from google.colab import userdata
from google import genai
from google.genai import types
from IPython.display import Image, Markdown, display

# API 클라이언트 초기화
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("✅ Gemini API Client 초기화 성공!")
except userdata.SecretNotFoundError:
    print("❌ 에러: 왼쪽 열쇠 아이콘(보안 비밀) 탭에서 'GEMINI_API_KEY'를 등록하고 액세스를 허용해주세요.")

# 사용할 모델 ID 지정
IMAGE_MODEL_ID = "gemini-3.1-flash-image-preview"
TEXT_MODEL_ID = "gemini-3-flash-preview"

## 제2장. 텍스트 기반 기본 이미지 생성 (Text-to-Image)

In [ ]:
prompt = "A diagram illustrating visual proof of the Pythagorean theorem."

print("⏳ 이미지를 생성하는 중입니다. (약 10~30초 소요)...")

response = client.models.generate_content(
    model=IMAGE_MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=["IMAGE"],
        image_config=types.ImageConfig(
            aspect_ratio="16:9", # 16:9 비율 지정
        ),
    ),
)

# 결과 출력
for part in response.candidates[0].content.parts:
    if part.inline_data:
        display(Image(data=part.inline_data.data, width=600))

## 제3장. LLM을 활용한 프롬프트 고도화 (Prompt Enhancement)

In [ ]:
# 제어하고 싶은 핵심 키워드
keywords = [
    "a simple slide",
    "explaining visual proof of the Pythagorean theorem",
    "white background",
    "eye-level shot",
    "white light",
    "minimalist"
]

# 텍스트 모델에게 프롬프트 확장을 지시하는 메타 프롬프트
gemini_prompt = f"""Your task is to expand the following keywords into a single, high-fidelity,
descriptive prompt for image generation. Every single keyword MUST be included.
Output ONLY the final prompt string, without any introduction or explanation.

Mandatory Keywords: {", ".join(keywords)}"""

response_text = client.models.generate_content(
    model=TEXT_MODEL_ID,
    contents=gemini_prompt,
)

image_prompt = response_text.text
display(Markdown(f"**✨ 고도화된 프롬프트:**\n> {image_prompt}"))

## 제4장 & 5장. 멀티모달 제어, 스타일 참조 및 결과 저장

In [ ]:
from PIL import Image as PILImage

# 1. 더미 레퍼런스 이미지 생성 (실제 사용 시 원하는 이미지 경로로 변경)
temp_img = PILImage.new('RGB', (800, 450), color = '#f4f6f9')
temp_img.save("slide-template.png")
print("⏳ 레퍼런스 이미지를 참조하여 최종 슬라이드를 생성 중입니다...")

with open("slide-template.png", "rb") as f:
    reference_image_bytes = f.read()

# 2. Image + 고도화된 프롬프트 결합하여 최종 생성
final_response = client.models.generate_content(
    model=IMAGE_MODEL_ID,
    contents=[
        types.Part.from_bytes(
            data=reference_image_bytes,
            mime_type="image/png",
        ),
        f"""Use this reference image as a general style guide to generate a lecture slide.
        Include only the necessary sections from the template. Use the guide as a general
        style formatting template instead of a strict outline. The slide image should be
        the entire image.

        Slide concept: {image_prompt}"""
    ],
    config=types.GenerateContentConfig(
        response_modalities=['IMAGE'],
        image_config=types.ImageConfig(
            aspect_ratio="16:9",
        ),
    ),
)

# 3. 결과 출력 및 다운로드 가능한 형태로 저장
for part in final_response.candidates[0].content.parts:
    if part.inline_data:
        final_image_data = part.inline_data.data
        display(Image(data=final_image_data, width=600))

        save_path = "final-lecture-slide.png"
        with open(save_path, "wb") as img_file:
            img_file.write(final_image_data)

        display(Markdown(f"**✅ 파일 저장 완료:** 이미지 파일이 Colab 좌측의 폴더 아이콘 내 `{save_path}`에 생성되었습니다. 우클릭하여 직접 다운로드 할 수 있습니다."))